# Data Wrangling 2.2 Solutions

In [ ]:
import math
import numpy as np
import pandas as pd

import psycopg2

import json

import csv

from datetime import datetime as dt

from IPython.display import display, HTML


from jellyfish import soundex, levenshtein_distance

from fuzzywuzzy import fuzz

from fuzzywuzzy import process as fuzz_process


In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

In [ ]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

## You try it - Repeat for bad customer first names;

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select cu1.stage_id,
       cu1.first_name as stage_first_name,
       cu2.first_name as customer_first_name
from stage_3_customers as cu1
     join customers as cu2
         on cu1.customer_id::numeric = cu2.customer_id
where cu1.first_name <> cu2.first_name
order by cu1.stage_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [ ]:

connection.rollback()

query = """

select cu1.stage_id,
       cu1.first_name as stage_first_name,
       cu2.first_name as customer_first_name
from stage_3_customers as cu1
     join customers as cu2
         on cu1.customer_id::numeric = cu2.customer_id
where cu1.first_name <> cu2.first_name
order by cu1.stage_id
;

"""
    
cursor.execute(query)

connection.rollback()
    
rows = cursor.fetchall()
    
for row in rows:
        print("---------------------------------------------------------")
        print("Wrong:", row[1], "soundex", soundex(row[1]))
        print("Right:", row[2], "soundex", soundex(row[2]))
        print("Levenshtein Distance:", levenshtein_distance(row[1], row[2]))
        print("Fuzzy: ratio:", fuzz.ratio(row[1], row[2]))
        print("Fuzzy: partial ratio:", fuzz.partial_ratio(row[1], row[2]))
        print("Fuzzy: token sort ratio:", fuzz.partial_ratio(row[1], row[2]))
        

In [ ]:

connection.rollback()

query = """

select distinct first_name
from customers
order by 1
;

"""
    
cursor.execute(query)

connection.rollback()
    
rows = cursor.fetchall()
    
first_name_list = []
    
for row in rows:
        first_name_list.append(row[0])
        
print(first_name_list[:100])

In [ ]:

connection.rollback()

query = """

select cu1.stage_id,
       cu1.first_name as stage_first_name,
       cu2.first_name as customer_first_name
from stage_3_customers as cu1
     join customers as cu2
         on cu1.customer_id::numeric = cu2.customer_id
where cu1.first_name <> cu2.first_name
order by cu1.stage_id
;


"""
    
cursor.execute(query)

connection.rollback()
    
rows = cursor.fetchall()
    
for row in rows:
        print("---------------------------------------------------------")
        print("Wrong:", row[1], "soundex", soundex(row[1]))
        print("Right:", row[2], "soundex", soundex(row[2]))
        print("Levenshtein Distance:", levenshtein_distance(row[1], row[2]))
        print("Fuzzy top 5 choices:", fuzz_process.extract(row[1], first_name_list, limit=5))
        

## You try it - Find the duplicate sales 

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select sa.store_id,
       sa.sale_id,
       sa.customer_id,
       sa.sale_date,
       sa.total_amount,
       count(*) number_of_duplicates
from stage_3_sales as sa
group by sa.store_id, sa.sale_id, sa.customer_id, sa.sale_date, sa.total_amount
having count(*) > 1
order by store_id, sale_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

with a as (

        select sa.store_id,
               sa.sale_id,
               sa.customer_id,
               sa.sale_date,
               sa.total_amount
        from stage_3_sales as sa
        group by sa.store_id, sa.sale_id, sa.customer_id, sa.sale_date, sa.total_amount
        having count(*) > 1

    )

select *
from stage_3_sales
where (store_id, sale_id) in (select store_id, sale_id from a)
order by stage_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - Find the missing customer first names  

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stage_3_customers
where first_name is null
order by customer_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - Find the outliers for line item quantity

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stage_3_line_items
where quantity::numeric >= 10
order by store_id, sale_id, line_item_id
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)